In [1]:
!pip install gradio cohere python-dotenv scikit-learn

In [2]:
!pip install --upgrade huggingface_hub

In [1]:
import os
import sys

# --- KEEP THE GRADIO BYPASS ACTIVE ---
import huggingface_hub
if not hasattr(huggingface_hub, 'HfFolder'):
    class MockHfFolder:
        @staticmethod
        def get_token(): return None
        @staticmethod
        def save_token(token): pass
    huggingface_hub.HfFolder = MockHfFolder

import gradio_client
if hasattr(gradio_client.utils, 'json_schema_to_python_type'):
    gradio_client.utils.json_schema_to_python_type = lambda schema: "Any"

os.environ["GRADIO_ANALYTICS_ENABLED"] = "False"

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import gradio as gr
from PIL import Image

print("🔥 Environment ready. Preparing Balanced Deep Learning Pipeline...")

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

# 1. Load and Balance the Dataset for Real Learning
def load_balanced_data(csv_path, img_dir):
    df = pd.read_csv(csv_path)
    df['img_filename'] = df['image_name'] + '.jpg'
    df['full_path'] = df['img_filename'].apply(lambda x: os.path.join(img_dir, x))
    
    # Filter valid files
    df = df[df['full_path'].apply(os.path.exists)].reset_index(drop=True)
    
    # Separate classes to handle imbalance
    benign_df = df[df['target'] == 0]
    malignant_df = df[df['target'] == 1]
    
    # Take a meaningful sample from both classes (e.g., 1500 each)
    n_samples = min(1500, len(benign_df), len(malignant_df))
    
    balanced_df = pd.concat([
        benign_df.sample(n=n_samples, random_state=42),
        malignant_df.sample(n=n_samples, random_state=42)
    ]).sample(frac=1, random_state=42).reset_index(drop=True) # Shuffle completely
    
    print(f"⚖️ Created a balanced subset of {len(balanced_df)} images ({n_samples} Benign, {n_samples} Malignant).")
    return balanced_df

def parse_image_and_label(filename, label):
    image_string = tf.io.read_file(filename)
    image = tf.image.decode_jpeg(image_string, channels=3)
    image = tf.image.resize(image, IMG_SIZE)
    image = image / 255.0  # Normalize
    return image, label

csv_path = 'train_concat.csv'
img_dir = 'train'
train_df = load_balanced_data(csv_path, img_dir)

if train_df is not None:
    # Build robust data loader
    dataset = tf.data.Dataset.from_tensor_slices((train_df['full_path'].values, train_df['target'].values))
    dataset = dataset.map(parse_image_and_label).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    
    # 2. A slightly deeper model to catch dynamic features
    model = keras.Sequential([
        layers.Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 3)),
        layers.Conv2D(32, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.GlobalAveragePooling2D(),
        layers.Dense(32, activation='relu'),
        layers.Dense(1, activation='sigmoid')
    ])
    
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    
    print("\n🚀 Training model on balanced features (This will take around 1-2 mins)...")
    # Training for 3 epochs so weights actually adjust to changes
    model.fit(dataset, epochs=3, verbose=1)
    
    model.save('skin_cancer_model.h5')
    print("🎉 Smart Model saved successfully as 'skin_cancer_model.h5'!")
    
    # 3. Dynamic Prediction Engine
    def predict_and_triage(image):
        try:
            if image is None: return "No image loaded.", "Upload Target Asset."
            img = Image.fromarray(image.astype('uint8')).convert('RGB').resize(IMG_SIZE)
            img_array = np.array(img) / 255.0
            img_array = np.expand_dims(img_array, axis=0)
            
            # Predict
            prediction = model.predict(img_array, verbose=0)[0][0]
            
            # Thresholding logic
            predicted_class = 1 if prediction > 0.5 else 0
            confidence = prediction * 100 if predicted_class == 1 else (1 - prediction) * 100
            
            result_text = f"### **System Classification:** {'🔴 MALIGNANT (Cancer Detected)' if predicted_class == 1 else '🟢 BENIGN (Normal Lesion)'}\n"
            result_text += f"### **Neural Confidence Score:** {confidence:.1f}%\n"
            result_text += f"### **Urgency Priority:** {'🚨 CRITICAL - Immediate Specialist Intervention' if predicted_class == 1 else '📅 LOW - Routine Clinical Review'}"
            
            triage_note = f"📋 **DermAI Nurse Triage Report Summary:**\nTarget tissue matrix analyzed dynamically. Tracking index displays evaluation confidence of {confidence:.1f}%."
            return result_text, triage_note
        except Exception as e:
            return f"Pipeline Error: {str(e)}", "Inference Failed."

    # 4. Launch UI Workspace
    print("\n🖥️ Booting up local Gradio Server...")
    with gr.Blocks(title="DermAI Workspace", theme=gr.themes.Soft()) as demo:
        gr.Markdown("# 🩺 DermAI: Skin Cancer Triage Assistant Workspace")
        with gr.Row():
            with gr.Column():
                input_image = gr.Image(label="Dermoscopic Skin Lesion Input", type="numpy")
                submit_btn = gr.Button("🔍 Run Analysis Pipeline", variant="primary", size="lg")
            with gr.Column():
                result_output = gr.Markdown(label="System Classifications")
                triage_output = gr.Markdown(label="📋 Generated Nurse Triage Document")
        
        submit_btn.click(fn=predict_and_triage, inputs=input_image, outputs=[result_output, triage_output])
    
    demo.launch(inline=True, share=True)
else:
    print("❌ Dataset Error: Verification failed.")

F:\conda_envs\isl3\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🔥 Environment ready. Preparing Balanced Deep Learning Pipeline...
⚖️ Created a balanced subset of 3000 images (1500 Benign, 1500 Malignant).

🚀 Training model on balanced features (This will take around 1-2 mins)...
Epoch 1/3
94/94 [==============================] - 128s 1s/step - loss: 0.6368 - accuracy: 0.6230
Epoch 2/3
94/94 [==============================] - 118s 1s/step - loss: 0.4699 - accuracy: 0.7810
Epoch 3/3
94/94 [==============================] - 121s 1s/step - loss: 0.4396 - accuracy: 0.8040
🎉 Smart Model saved successfully as 'skin_cancer_model.h5'!

🖥️ Booting up local Gradio Server...
Running on local URL:  http://127.0.0.1:7860
Running on public URL: https://bd92714228b614f6d5.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)
